In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
import seaborn as sns
import matplotlib.pyplot as plt


In [72]:
# 1. Load Data
df = pd.read_csv('/content/loan_approval_dataset (1).csv')
print("Original Data Shape:", df.shape)
print(df.head())

Original Data Shape: (4269, 13)
   loan_id   no_of_dependents      education  self_employed   income_annum  \
0        1                  2       Graduate             No        9600000   
1        2                  0   Not Graduate            Yes        4100000   
2        3                  3       Graduate             No        9100000   
3        4                  3       Graduate             No        8200000   
4        5                  5   Not Graduate            Yes        9800000   

    loan_amount   loan_term   cibil_score   residential_assets_value  \
0      29900000          12           778                    2400000   
1      12200000           8           417                    2700000   
2      29700000          20           506                    7100000   
3      30700000           8           467                   18200000   
4      24200000          20           382                   12400000   

    commercial_assets_value   luxury_assets_value   bank_asset_val

In [74]:
# 2. Basic Info

print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())


Data Types:
 loan_id                       int64
 no_of_dependents             int64
 education                   object
 self_employed               object
 income_annum                 int64
 loan_amount                  int64
 loan_term                    int64
 cibil_score                  int64
 residential_assets_value     int64
 commercial_assets_value      int64
 luxury_assets_value          int64
 bank_asset_value             int64
 loan_status                 object
dtype: object

Missing Values:
 loan_id                      0
 no_of_dependents            0
 education                   0
 self_employed               0
 income_annum                0
 loan_amount                 0
 loan_term                   0
 cibil_score                 0
 residential_assets_value    0
 commercial_assets_value     0
 luxury_assets_value         0
 bank_asset_value            0
 loan_status                 0
dtype: int64


In [75]:
# Separate numerical and categorical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns
target_col = 'Accident'  # Corrected target column

# Numerical: Replace with mean, only if there are missing values
for col in num_cols:
    if df[col].isnull().any():
        num_imputer = SimpleImputer(strategy='mean')
        df[col] = num_imputer.fit_transform(df[[col]])

# Categorical: Replace with mode, only if there are missing values
for col in cat_cols:
    if df[col].isnull().any():
        cat_imputer = SimpleImputer(strategy='most_frequent')
        # Fix: Flatten the 2D output of fit_transform before assigning to a single column
        df[col] = cat_imputer.fit_transform(df[[col]]).ravel()

# Check missing values after handling
print(df.isnull().sum())

loan_id                      0
 no_of_dependents            0
 education                   0
 self_employed               0
 income_annum                0
 loan_amount                 0
 loan_term                   0
 cibil_score                 0
 residential_assets_value    0
 commercial_assets_value     0
 luxury_assets_value         0
 bank_asset_value            0
 loan_status                 0
dtype: int64


In [76]:
# 4. Encode Categorical Variables
le = LabelEncoder()

for col in cat_cols:
    if df[col].nunique() == 2:
        df[col] = le.fit_transform(df[col])

# One-hot Encoding (for more than 2 categories)
df = pd.get_dummies(
    df,
    columns=[col for col in cat_cols if df[col].nunique() > 2],
    drop_first=True
)

In [77]:
# 5. Outlier Detection and Treatment (using IQR)
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = np.where(df[col] < lower, lower, df[col])
    df[col] = np.where(df[col] > upper, upper, df[col])


In [78]:
# 6. Feature Scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])


In [79]:
target_col = ' loan_status'
if target_col in df.columns:
    X = df.drop(target_col, axis=1)
    y = df[target_col]

    bestfeatures = SelectKBest(score_func=f_classif, k=10)

    fit = bestfeatures.fit(X, y)

    selected_features = X.columns[fit.get_support()]

    df = df[selected_features.to_list() + [target_col]]

    print("\nSelected Features:", selected_features.to_list())


Selected Features: ['loan_id', ' no_of_dependents', ' income_annum', ' loan_amount', ' loan_term', ' cibil_score', ' residential_assets_value', ' commercial_assets_value', ' luxury_assets_value', ' bank_asset_value']


In [88]:
X = np.asarray(df[[' no_of_dependents',' income_annum',' loan_amount',' cibil_score']])
y = np.asarray(df[' loan_status'])
X = StandardScaler().fit(X).transform(X)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size = 0.3, random_state = 4)
print ('Train set:', X_train.shape, y_train.shape)
print ('Test set:', X_test.shape, y_test.shape)

Train set: (2988, 4) (2988,)
Test set: (1281, 4) (1281,)


In [93]:
# 1. Logistic Regression
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy:", model.score(X_test, y_test))

Accuracy: 0.9172521467603435


In [96]:
# 2. K-Nearest Neighbors
from sklearn.neighbors import KNeighborsClassifier
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy:", model.score(X_test, y_test))


Accuracy: 0.9188134270101483


In [97]:
# 3. Decision Tree
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy:", model.score(X_test, y_test))

Accuracy: 0.921935987509758


In [98]:
# 4. Random Forest
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy:", model.score(X_test, y_test))

Accuracy: 0.9430132708821234


In [99]:
# 5. Support Vector Machine
from sklearn.svm import SVC
model = SVC()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy:", model.score(X_test, y_test))

Accuracy: 0.9328649492583919


In [100]:
# 6. Gradient Boosting
from sklearn.ensemble import GradientBoostingClassifier
model = GradientBoostingClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy:", model.score(X_test, y_test))

Accuracy: 0.9430132708821234


In [101]:
# 7. Naive Bayes
from sklearn.naive_bayes import GaussianNB
model = GaussianNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy:", model.score(X_test, y_test))

Accuracy: 0.921935987509758
